In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
from scipy.spatial.distance import cosine

# 1. Load the pre-trained model
# We set include_top=False because we don't want it to classify "dog" or "cat".
# We just want the raw mathematical features (pooling='avg' flattens it to a 1D array).
print("Loading AI Model...")
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
print("Model Loaded!")

def generate_content_dna(img_path):
    """Takes an image and returns its 1280-dimensional Content DNA hash."""
    # Load image and resize to what the model expects
    img = image.load_img(img_path, target_size=(224, 224))

    # Convert to array and add a batch dimension
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)

    # Preprocess the image exactly how the model expects it
    x = preprocess_input(x)

    # Extract the features (This is your DNA!)
    dna_vector = base_model.predict(x)
    return dna_vector[0]

def compare_dna(dna1, dna2):
    """Calculates how similar two DNA vectors are (1.0 = identical, 0.0 = completely different)"""
    # Using Cosine Similarity to compare the mathematical vectors
    similarity = 1 - cosine(dna1, dna2)
    return similarity

Loading AI Model...


/tmp/ipykernel_4632/1566231786.py:12: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model Loaded!


In [3]:
# Generate DNA for your test images
dna_a = generate_content_dna("/content/image_a.jpg")
dna_b = generate_content_dna("/content/image_b.png") # The pirated, edited version
dna_c = generate_content_dna("/content/image_c.png") # Completely different image

print("--- Similarity Results ---")
print(f"Original vs. Pirated (Should be high, near 0.9+): {compare_dna(dna_a, dna_b):.4f}")
print(f"Original vs. Random (Should be low): {compare_dna(dna_a, dna_c):.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
--- Similarity Results ---
Original vs. Pirated (Should be high, near 0.9+): 0.9918
Original vs. Random (Should be low): 0.3926


In [5]:
import cv2
import numpy as np

def extract_video_dna(video_path, frames_per_second_to_extract=1):
    """Reads a video, extracts frames at a set rate, and returns a sequence of DNA vectors."""
    print(f"Opening video: {video_path}")
    cap = cv2.VideoCapture(video_path)

    # Get the video's actual framerate
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        print("Error: Could not read video. Check the file path.")
        return []

    frame_interval = int(fps / frames_per_second_to_extract)
    video_dna_sequence = []
    frame_count = 0
    extracted_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break # Video is over

        # Only process 1 frame every 'frame_interval'
        if frame_count % frame_interval == 0:
            # OpenCV loads videos in BGR color space, AI models expect RGB
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Resize for MobileNetV2
            resized_frame = cv2.resize(rgb_frame, (224, 224))

            # Format and predict
            x = np.expand_dims(resized_frame, axis=0)
            x = preprocess_input(x.astype(np.float32))

            # Generate the DNA for this specific second of video (verbose=0 hides the progress bars)
            frame_dna = base_model.predict(x, verbose=0)[0]
            video_dna_sequence.append(frame_dna)
            extracted_count += 1

        frame_count += 1

    cap.release()
    print(f"Done! Extracted a sequence of {extracted_count} DNA vectors.")
    return video_dna_sequence

# --- HOW TO TEST THIS ---
# 1. Upload a short 5-10 second video to Colab (name it 'test_clip.mp4')
# 2. Run:
my_video_dna = extract_video_dna("test_clip.mp4")

Opening video: test_clip.mp4
Done! Extracted a sequence of 7 DNA vectors.
